# Tuned-lens read robustness (Experiment 5)
The logit lens is optimistic at intermediate depth. We replicate intermediate read with a tuned lens on 1–2 models. If the dissociation persists, the 'intermediate read is a readout artifact later layers correctly resolve' objection is answered empirically. Falls back to the read-definition sweep + hedge if pretrained lenses are unavailable.

In [ ]:
# --- environment (pins matching the pipeline) ---
# transformers==4.46.2  numpy==1.26.4  ; PyTorch nightly cu128 on newer instances.
import os, gc, json, math, pathlib
import numpy as np, torch
from tqdm.auto import tqdm
import rw_core as rc          # tested core (rw_core_smoketest.py: 19/19)
import rw_modelio as mio      # model IO / hooks / generation

ART = pathlib.Path(os.environ.get("RW_ART", "artifacts")); ART.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def save(obj, name):
    p = ART / name
    np.savez_compressed(p, **obj) if name.endswith(".npz") else \
        p.write_text(json.dumps(obj, indent=2, default=float))
    print("saved", p)

def exists(name):  # skip-if-exists guard
    return (ART / name).exists()


In [ ]:
SUBSET = ["meta-llama/Llama-3.1-8B", "Qwen/Qwen2.5-7B"]
USE_TUNED_LENS = True
try:
    from tuned_lens import TunedLens
except Exception as e:
    print("tuned_lens unavailable:", e); USE_TUNED_LENS = False

In [ ]:
# Tuned-lens read: replace logit-lens mid-layer decode with the tuned affine
# probe, then recompute the priming margin and not-top-ranked rate.
from transformers import AutoModelForCausalLM, AutoTokenizer
def tuned_read_int(model, lens, tok, prompt, token_id, band):
    ids = tok(prompt, return_tensors="pt").to(DEVICE)
    hs = model(**ids, output_hidden_states=True).hidden_states
    pos = ids["input_ids"].shape[1]-1
    best = -1e9
    for l in band:
        logits = lens.forward(hs[l][0, pos], l-1)         # tuned affine at layer l
        best = max(best, float(torch.log_softmax(logits, -1)[token_id]))
    return best

rows = []
if USE_TUNED_LENS:
    for name in SUBSET:
        model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16,
                                                     device_map=DEVICE).eval()
        tok = AutoTokenizer.from_pretrained(name)
        lens = TunedLens.from_model_and_pretrained(model).to(DEVICE)
        band = mio.band_indices(model.config.num_hidden_layers)
        items = load_items(name)            # needs prompt, gold/decoy tok ids, write_rank
        for it in tqdm(items):
            it["read_int_gold"] = tuned_read_int(model, lens, tok, it["prompt"],
                                                 it["gold_first_tok"], band)
            it["read_int_decoys"] = [tuned_read_int(model, lens, tok, it["prompt"], d, band)
                                     for d in it["decoy_first_toks"]]
        tbl = rc.readwrite_table(items, criterion="hard")
        rows.append({"model": name, **tbl})
        del model, lens; gc.collect(); torch.cuda.empty_cache()
    save({"rows": rows}, "exp5_tunedlens.json"); print(rows)
else:
    print("Falling back: promote hard-decoy headline (NB 2) and hedge read as "
          "'decodability under this probe'. The 24-variant read sweep already "
          "shows the conclusion is stable across logit-lens definitions.")

Whatever the outcome, add to the methods: *intermediate read is decodability under a fixed probe (logit/tuned lens), not a claim about stored knowledge.* The tuned-lens replication is the clean reviewer defense; the read-definition sweep is the cheap one.